# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a workflow to load and explore the FAIR² dataset using the `mlcroissant` library. Follow each section to discover the structure, metadata, and example analysis on the data using only entity `@id` references.

### Dataset Source
The dataset is described by the [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and available for programmatic access.

In [ ]:
# Ensure `mlcroissant` is available
!pip install mlcroissant

## 1. Data Loading
Load metadata and dataset records with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display name and description
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets and each record set's fields and their IDs.

We use only `@id` references for entities, as specified.

In [ ]:
# List all record sets by @id and their fields by @id

record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
print("Record Sets @id(s):")
for rsid in record_set_ids:
    print(f"- {rsid}")

if not record_set_ids:
    # Try to extract record set IDs via dataset API if not found in metadata
    print("\nNo recordSet listed directly in metadata. Attempting to infer available recordSets...")
    inferred_recordset_ids = set()
    try:
        for rsi in dataset.record_sets:
            print(f"- {rsi['@id']}")
            inferred_recordset_ids.add(rsi['@id'])
        record_set_ids = list(inferred_recordset_ids)
    except Exception as e:
        print("Could not infer record sets. Please check dataset description directly.")

# For each record set, print their fields (columns) by @id
print("\nFields (columns) by record set @id:")
for rsid in record_set_ids:
    rs = dataset.record_set(rsid)
    field_ids = [field['@id'] for field in rs['field']]
    print(f"Record set {rsid}:")
    print("  Fields IDs:")
    for fid in field_ids:
        print(f"    - {fid}")

## 3. Data Extraction
Extract data from the available record set(s) into pandas DataFrames, using their `@id`s.

In [ ]:
# Prepare: list all available recordSet @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    # Fallback: collect inferred ones from dataset.record_sets
    record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
# Load data for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded: {record_set_id} ({len(df)} records, columns: {list(df.columns)})")

# Show columns of the first available record set
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nColumns for {example_rs_id}:\n{dataframes[example_rs_id].columns.tolist()}")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
In this section, we demonstrate filtering, normalization, and grouping, using only `@id` references for fields. 

**If you wish to use a different record set or field, replace the IDs accordingly.**

In [ ]:
# Use the first record set for EDA
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# List numeric-like columns as candidates
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
# Fallback if no numeric, select first column (may require manual editing)
if numeric_field_id is None:
    print('No numeric field auto-detected; please check dataset columns and adjust filtering logic.')
    numeric_field_id = df.columns[0]

# Apply thresholding
threshold = 10
try:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Pick another field (by @id) to group by
    group_candidate = None
    for col in df.columns:
        if col != numeric_field_id:
            group_candidate = col
            break

    if group_candidate:
        grouped_df = filtered_df.groupby(group_candidate)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_candidate}:")
        display(grouped_df.head())
    else:
        print('No grouping field found.')
except Exception as e:
    print(f"Unable to run filtering and normalization due to: {e}\nThis may be due to non-numeric field types or empty DataFrame.")

## 5. Visualization
We visualize the distribution of a numeric field or relationship between two fields, using their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Visualize the distribution of the numeric field in the first record set
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=10, alpha=0.7)
    plt.title(f'Distribution of field: {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Scatter plot with another available field
if group_candidate and numeric_field_id and group_candidate in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(7,5))
    plt.scatter(df[group_candidate], df[numeric_field_id], alpha=0.6)
    plt.title(f'{numeric_field_id} vs. {group_candidate}')
    plt.xlabel(group_candidate)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to programmatically load, inspect, and process data from a FAIR² clinical dataset using only `@id` references. You can adapt this template to any Croissant dataset by simply changing the schema URL and referring to entities using their stable identifiers.

Key steps included:
- Loading metadata and listing record sets/fields by `@id`.
- Extracting records into DataFrames for analysis.
- Filtering, normalization, and grouping using `@id`s only.
- Basic plotting for visual exploration.